<a href="https://colab.research.google.com/github/kumarjitc/trip-to-jupyter/blob/main/TextModeration-Playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Steps

1. Install And Setup
2. OpenAI & LLM Setup
3. Data Setup
4. Test The Model
5. Push to Git

##Notes
We will use the unitary/toxic-bert model, which is fine-tuned for detecting toxicity. Since I am broke. :D




##Install And Setup

In [ ]:
# use capture to hide the long output
%%capture log_pip_install_openai
import sys

!pip uninstall -y mistralai # Force uninstall any existing mistralai package
!pip install openai
!pip install gradio
!pip install huggingface_hub
!pip install mistralai==0.4.2 # Install the specifically requested version
!pip install opendatasets

In [ ]:
import openai
import gradio
import huggingface_hub
from mistralai.client import MistralClient
from google.colab import userdata

In [ ]:
import pkg_resources

print(f"openai version: {openai.__version__}")
print(f"gradio version: {gradio.__version__}")
print(f"huggingface_hub version: {huggingface_hub.__version__}")
try:
    mistralai_version = pkg_resources.get_distribution('mistralai').version
    print(f"mistralai version: {mistralai_version}")
except pkg_resources.DistributionNotFound:
    print("mistralai not found")


In [ ]:
import pkg_resources
try:
    mistralai_version = pkg_resources.get_distribution('mistralai').version
    print(f"mistralai version: {mistralai_version}")
except pkg_resources.DistributionNotFound:
    print("mistralai not found")

###Install And Setup Pluto

####Start - Review Later

In [ ]:
# prompt: print git version

!git --version

In [ ]:
# prompt: install lfs and track large file *.pkl

# note this optional for git to upload/push large file like the inference engine, *.pkl file
!apt -y install git-lfs
!git lfs install
!git lfs track "*.pkl"

In [ ]:
# prompt: clone https://github.com/duchaba/pluto_happy

fname = 'https://github.com/duchaba/pluto_happy'
!git clone {fname}

In [ ]:
# prompt: list the content of pluto_happy directory

!ls -la pluto_happy

In [ ]:
# prompt: pip install requriements.txt

%%capture log_pip_install
fname = 'pluto_happy/requirements.txt'
!pip install -r {fname}

In [ ]:
# print the log file is failed
# log_pip_install.show()

In [ ]:
# prompt: run the pluto_happy/pluto.py

fname = 'pluto_happy/pluto.py'
%run {fname}

In [ ]:
# %%write -a app.py
# prompt: create a new class Pluto_Happy and name it monty

monty = Pluto_Happy('Monty, Monty Said!')

In [ ]:
# not_write -a app.py
# prompt: None.

# print out my environments
monty.fname_requirements = 'pluto_happy/requirements.txt'
monty.print_info_self()

In [ ]:
# prompt: print all monty functions doc

help(monty)

####End - Review Later

##Setup LLM

###Set Keys To Env Vars

In [ ]:
os.environ['openai_key'] = userdata.get('OPENAPI_KEY')
print("Keys set to environment variables.")

In [ ]:
# %%write app.py
# Prompt: None
# replace the "getenv()" with your key string

import os
monty._openai_key=os.getenv('openai_key')
monty._github_key=os.getenv('github_token') # Corrected: used 'github_token' instead of 'github_key'
monty._huggingface_key=os.getenv('huggingface_key')
monty._kaggle_key=os.getenv('kaggle_key')

### Free Text Moderation with Hugging Face `transformers`

We will use the `unitary/toxic-bert` model, which is fine-tuned for detecting toxicity. First, ensure the `transformers` library is installed.

In [ ]:
# Install the transformers library if not already installed
!pip install transformers torch

### Note on Hugging Face Model Usage

*   **Resource Usage:** Running these models locally will consume Colab's RAM and CPU/GPU resources. For very large datasets, you might need a GPU runtime.
*   **Model Specificity:** Different models are trained for different types of moderation (e.g., general toxicity, hate speech, sentiment). Choose a model that best fits your specific moderation needs.
*   **Thresholding:** The output probabilities need to be interpreted using thresholds that you define based on your application's requirements. A probability of 0.5 is a common starting point but may need adjustment.

In [ ]:
import os
monty._openai_key=os.getenv('openai_key')
monty._github_key=os.getenv('github_key')
monty._huggingface_key=os.getenv('huggingface_key')
monty._kaggle_key=os.getenv('kaggle_key')

###Get AI Client

In [ ]:
# help(ai_client.moderations.create)

##Setup Data

In [ ]:
# prompt install opendatasets
!pip install opendatasets
import opendatasets

In [ ]:
# prompt: Write function with inline documentation to download dataset from Kaggle website using opendatasets lib.

# I add line @add_method for put the function to pluto (Note: this is optional)
@add_method(Pluto_Happy)
def fetch_kaggle_dataset(self,dataset_name, path_to_save):

  """
  Downloads a dataset from Kaggle website using opendatasets library.

  Args:
    dataset_name: (str) The name of the dataset to download.
    path_to_save: (str) The path where the dataset will be saved.

  Returns:
    None
  """

  try:
    # Check if the dataset already exists
    if os.path.exists(path_to_save):
      print(f'Dataset {dataset_name} already exists.')
      return

    # Download the dataset
    print(f'Downloading dataset {dataset_name}...')
    opendatasets.download(dataset_name, path_to_save)
    print(f'Dataset {dataset_name} downloaded successfully.')

  except Exception as e:
    print(f'Error downloading dataset {dataset_name}: {e}')
  return None

In [ ]:
# prompt: use monty.fetch_kaggle_dataset to download https://www.kaggle.com/competitions/jigsaw-toxic-severity-rating

fname = 'https://www.kaggle.com/competitions/jigsaw-toxic-severity-rating'
monty.fetch_kaggle_dataset(fname,'kaggle')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Ensure tokenizer and model are loaded only once
if 'tokenizer' not in locals() or 'model' not in locals():
    tokenizer = AutoTokenizer.from_pretrained("unitary/toxic-bert")
    model = AutoModelForSequenceClassification.from_pretrained("unitary/toxic-bert")

def moderate_text_hf(text):
    """
    Moderates the given text using the unitary/toxic-bert Hugging Face model.

    Args:
        text (str): The input text to moderate.

    Returns:
        dict: A dictionary containing toxicity scores for various categories.
    """
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Perform inference
    with torch.no_grad():
        outputs = model(**inputs)

    # Get predicted probabilities
    probabilities = torch.sigmoid(outputs.logits)

    # Map probabilities to specific labels
    labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    results = {label: prob.item() for label, prob in zip(labels, probabilities[0])}

    return results

# Example usage:
example_text = "I really like this movie, it's fantastic!"
moderation_scores = moderate_text_hf(example_text)
print(f"Text: '{example_text}'")
print(f"Moderation Scores: {moderation_scores}")

example_toxic_text = "You are an idiot and should shut up."
moderation_scores_toxic = moderate_text_hf(example_toxic_text)
print(f"\nText: '{example_toxic_text}'")
print(f"Moderation Scores: {moderation_scores_toxic}")


In [ ]:
# prompt: list the data in kaggle/jigsaw-toxic-severity-rating
!ls -la kaggle/jigsaw-toxic-severity-rating

###Import to Pandas

In [ ]:
# prompt: load fname csv into dataframe

import pandas
fname = '/content/kaggle/jigsaw-toxic-severity-rating/validation_data.csv'
monty.df_toxic_data = pandas.read_csv(fname)
monty.df_toxic_data.head(2)

###Clean Data

In [ ]:
# prompt: replace \n with space in the df_toxic_data column less_toxic and more_toxic

monty.df_toxic_data['less_toxic'] = monty.df_toxic_data['less_toxic'].str.replace('\n', ' ')
monty.df_toxic_data['more_toxic'] = monty.df_toxic_data['more_toxic'].str.replace('\n', ' ')

In [ ]:
# prompt: replace \n with space in the df_toxic_data column less_toxic and more_toxic

monty.df_toxic_data['less_toxic'] = monty.df_toxic_data['less_toxic'].str.replace('http', 'tthp')
monty.df_toxic_data['more_toxic'] = monty.df_toxic_data['more_toxic'].str.replace('http', 'tthp')
monty.df_toxic_data['less_toxic'] = monty.df_toxic_data['less_toxic'].str.replace('.com', '.no')
monty.df_toxic_data['more_toxic'] = monty.df_toxic_data['more_toxic'].str.replace('.com', '.no')
monty.df_toxic_data['less_toxic'] = monty.df_toxic_data['less_toxic'].str.replace('.org', '.no')
monty.df_toxic_data['more_toxic'] = monty.df_toxic_data['more_toxic'].str.replace('.org', '.no')

In [ ]:
# prompt: replace any non-printing character with space in the df_toxic_data column less_toxic and more_toxic

monty.df_toxic_data['less_toxic'] = monty.df_toxic_data['less_toxic'].str.replace('[^\\x00-\\x7F]', ' ')
monty.df_toxic_data['more_toxic'] = monty.df_toxic_data['more_toxic'].str.replace('[^\\x00-\\x7F]', ' ')

In [ ]:
# prompt: set panda row display to be 250 character long

pandas.set_option('display.max_colwidth', 550)
monty.df_toxic_data.sample(2)

In [ ]:
monty.df_toxic_data.info()

###Save Data

In [ ]:
# prompt: write monty.df_toxic_data dataframe to csv file

monty.df_toxic_data.to_csv('toxic_data.csv', index=False)

In [ ]:
# %%write -a app.py

# fname = 'toxic_data.csv'
# monty.df_toxic_data = pandas.read_csv(fname)

###Analyse Data
- Count average word size of more_toxic.
- Count average word size of less_toxic.
- Plot histogram
- Report statistic
- Draw word cloud

In [ ]:
# prompt: create a new column in monty.df_toxic_data dataframe with the word count from "less_toxic" column

monty.df_toxic_data['less_toxic_word_count'] = monty.df_toxic_data['less_toxic'].apply(lambda x: len(x.split()))
monty.df_toxic_data.head(2)

In [ ]:
# prompt: create a new column in monty.df_toxic_data dataframe with the word count from "less_toxic" column

monty.df_toxic_data['more_toxic_word_count'] = monty.df_toxic_data['more_toxic'].apply(lambda x: len(x.split()))
monty.df_toxic_data.sample(2)

In [ ]:
# prompt: find the sum of "less_toxic_word_count"

lcount = monty.df_toxic_data['less_toxic_word_count'].sum()
mcount = monty.df_toxic_data['more_toxic_word_count'].sum()
print(lcount + mcount)

####Histogram

In [ ]:
# prompt: using pandas to draw the histogram of "less_toxic_word_count"

x = monty.df_toxic_data['less_toxic_word_count'].plot.hist(bins=10,
  title='Less Toxic Word Count Histogram')

In [ ]:
x = monty.df_toxic_data['more_toxic_word_count'].plot.hist(bins=10,
  title='More Toxic Word Count Histogram')

####Statistics

In [ ]:
# prompt: print the max, min, mean, and std of column "les_toxic_word_count"

max_value = monty.df_toxic_data['less_toxic_word_count'].max()
min_value = monty.df_toxic_data['less_toxic_word_count'].min()
mean_value = monty.df_toxic_data['less_toxic_word_count'].mean()
std_value = monty.df_toxic_data['less_toxic_word_count'].std()

print(f"Max: {max_value}, Min: {min_value}, Mean: {mean_value}, Std: {std_value}")


In [ ]:
# prompt: print the max, min, mean, and std of column "les_toxic_word_count"

max_value = monty.df_toxic_data['more_toxic_word_count'].max()
min_value = monty.df_toxic_data['more_toxic_word_count'].min()
mean_value = monty.df_toxic_data['more_toxic_word_count'].mean()
std_value = monty.df_toxic_data['more_toxic_word_count'].std()

print(f"Max: {max_value}, Min: {min_value}, Mean: {mean_value}, Std: {std_value}")

####Word Cloud

In [ ]:
import wordcloud
# help(wordcloud.WordCloud)

In [ ]:
# prompt: write a Python function with documentation for drawing a word cloud plot for a dataframe 'less_toxic_word_count'.

from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

def generate_wordcloud(df, column, title="Word Cloud"):
    """
    Generate a word cloud from text data in a specified DataFrame column.

    Args:
    df (pd.DataFrame): The DataFrame containing the text data.
    column (str): The name of the column containing the text data.

    Returns:
    None
    """
    # Ensure the column exists in the DataFrame
    if column not in df.columns:
        print(f"The column {column} does not exist in the DataFrame.")
        return

    # Combine all the text from the column into a single string
    text = ' '.join(df[column].astype(str).values)

    # create special word stops
    my_stop_words = {'page', 'will', 'one', 'edit', 'article', 'know', 'way', 'say'}
    combined_set = STOPWORDS.union(my_stop_words)

    # Create a WordCloud object and generate the wordcloud
    wordcloud = WordCloud(background_color='white', width=800, height=800,
      max_words=300, stopwords=combined_set).generate(text)

    # Display the generated wordcloud
    plt.figure(figsize=(8,8),facecolor = None)
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.title(title)
    plt.tight_layout(pad=0)
    plt.show()

    # Save the wordcloud to a file
    wordcloud.to_file('wordcloud.png')
    return

# Example usage:-
# generate_wordcloud(result_df, 'Article_Text')

In [ ]:
generate_wordcloud(monty.df_toxic_data, "less_toxic", "Less Toxic Word Cloud")

In [ ]:
generate_wordcloud(monty.df_toxic_data, "more_toxic", "More Toxic Word Cloud")

In [ ]:
# view word not count
wordcloud.STOPWORDS

In [ ]:
my_stop_words = {'page', 'will', 'one', 'edit', 'article', 'know', 'way', 'say'}

In [ ]:
# prompt: combine two sets, wordcloud.STOPWORDS and my_stop_words

combined_set = wordcloud.STOPWORDS.union(my_stop_words)

In [ ]:
combined_set

In [ ]:
# redraw them by re-run previous generate_wordcloud cell/command.

##Write API
### Moderation API Function for `unitary/toxic-bert`

This function encapsulates the moderation logic, allowing you to easily check the toxicity of any given text using the previously loaded `unitary/toxic-bert` model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("unitary/toxic-bert")
model = AutoModelForSequenceClassification.from_pretrained("unitary/toxic-bert")

# Example text for moderation
text_to_moderate = "You are stupid and I hate you!"

# Tokenize the input text
inputs = tokenizer(text_to_moderate, return_tensors="pt", padding=True, truncation=True)

# Perform inference
with torch.no_grad():
    outputs = model(**inputs)

# Get predicted probabilities and labels
probabilities = torch.sigmoid(outputs.logits)
predicted_class_id = torch.argmax(probabilities, dim=1).item()

# Get the label names (these depend on how the model was trained)
# For 'toxic-bert', the labels often include 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'
# You might need to inspect model.config.id2label for exact mapping

# This is a simplified approach, actual labels might vary.
# The 'toxic-bert' model typically outputs probabilities for multiple toxicity categories.
print(f"Text: '{text_to_moderate}'")
print(f"Predicted probabilities for toxicity categories: {probabilities}")

# For a more detailed output, you might need to map probabilities to specific labels
# Example of how you might interpret (requires knowing the model's label mapping):
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
results = {label: prob.item() for label, prob in zip(labels, probabilities[0])}
print(f"Detailed Toxicity Scores: {results}")

# A simple threshold for 'toxic'
if results['toxic'] > 0.5:
    print("This text is likely toxic.")
else:
    print("This text is likely not toxic.")

##Push To Github

#####Its really difficult to maintain the state here, so I updated the file in local and pushed via git cli. 